In [2]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd

CANDIDATES = [
    Path("train_genome_sweep_phase_helper.py"),
    Path("scripts/train_genome_sweep_phase_helper.py"),
    Path("/mnt/data/train_genome_sweep_phase_helper.py"),
]

MODULE_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert MODULE_PATH is not None, "Could not find train_genome_sweep_phase_helper.py"

spec = importlib.util.spec_from_file_location("sweep", MODULE_PATH)
sweep = importlib.util.module_from_spec(spec)

# Critical for dataclasses:
sys.modules["sweep"] = sweep

spec.loader.exec_module(sweep)


def decode_state(state_id):
    name = sweep.SWEEP_STATE_NAMES[int(state_id)]

    if name.startswith("U5_"):
        return name, "U5", int(name[-1]), None
    if name.startswith("U3_"):
        return name, "U3", int(name[-1]), None
    if name.startswith("C"):
        return name, "C", int(name[1]), int(name[2])
    if name.startswith("I"):
        return name, "I", int(name[1]), int(name[2])

    raise ValueError(name)

def gene_table(gene, start=0, end=None):
    if end is None:
        end = len(gene.dna)

    rows = []
    donor_set = set(gene.donor_positions)
    acceptor_set = set(gene.acceptor_positions)

    for pos in range(start, min(end, len(gene.dna))):
        state_id = int(gene.target_states[pos])
        state_name, region, g, p = decode_state(state_id)

        markers = []
        if pos == gene.start_codon_start:
            markers.append("START_ATG")
        if pos == gene.stop_codon_start:
            markers.append("STOP_START")
        if pos in donor_set:
            markers.append("DONOR")
        if pos in acceptor_set:
            markers.append("ACCEPTOR")

        rows.append({
            "pos": pos,
            "base": gene.dna[pos],
            "codon_from_here": gene.dna[pos:pos+3] if pos + 3 <= len(gene.dna) else "",
            "state": state_name,
            "region": region,
            "genomic_g": g,
            "cds_phase_p": p,
            "pos_mod3": pos % 3,
            "marker": ",".join(markers),
        })

    return pd.DataFrame(rows)

def show_junctions(gene, flank=12):
    print(f"DNA length: {len(gene.dna)}")
    print(f"Start ATG: {gene.start_codon_start}  {gene.dna[gene.start_codon_start:gene.start_codon_start+3]}")
    print(f"Stop start: {gene.stop_codon_start}  {gene.dna[gene.stop_codon_start:gene.stop_codon_start+3]}")
    print(f"Intron lengths: {gene.intron_lengths}")
    print()

    for j, (donor, acceptor, intron_len) in enumerate(
        zip(gene.donor_positions, gene.acceptor_positions, gene.intron_lengths)
    ):
        donor_state = decode_state(gene.target_states[donor])
        first_intron_state = decode_state(gene.target_states[donor + 1])
        last_intron_state = decode_state(gene.target_states[acceptor - 1])
        acceptor_state = decode_state(gene.target_states[acceptor])

        print(f"Junction {j}")
        print(f"  donor pos:    {donor}     state={donor_state[0]}")
        print(f"  intron len:   {intron_len}     mod3={intron_len % 3}")
        print(f"  first intron: {donor + 1}     state={first_intron_state[0]}")
        print(f"  last intron:  {acceptor - 1}     state={last_intron_state[0]}")
        print(f"  acceptor pos: {acceptor}     state={acceptor_state[0]}")
        print(f"  carried p through intron: {first_intron_state[3]} -> {last_intron_state[3]} -> {acceptor_state[3]}")
        print(f"  genomic g shift donor→acceptor: {donor_state[2]} -> {acceptor_state[2]}")
        print()

        display(gene_table(gene, donor - flank, acceptor + flank))

def make_example(intron_mod, seed=1):
    return sweep.generate_sweep_gene(
        utr5_length=24,
        coding_codons=24,
        utr3_length=24,
        exon_count=3,
        min_exon_length=9,
        min_intron_length=7,
        max_intron_length=20,
        intron_mod=intron_mod,
        seed=seed,
    )

# Try forced intron-length mod classes.
for intron_mod in ["0", "1", "2"]:
    print("=" * 80)
    print(f"EXAMPLE: intron_mod = {intron_mod}")
    print("=" * 80)

    gene = make_example(intron_mod=intron_mod, seed=42)
    show_junctions(gene, flank=8)

EXAMPLE: intron_mod = 0
DNA length: 153
Start ATG: 24  ATG
Stop start: 126  TAG
Intron lengths: (18, 15)

Junction 0
  donor pos:    43     state=C11
  intron len:   18     mod3=0
  first intron: 44     state=I22
  last intron:  61     state=I12
  acceptor pos: 62     state=C22
  carried p through intron: 2 -> 2 -> 2
  genomic g shift donor→acceptor: 1 -> 2



,pos,base,codon_from_here,state,region,genomic_g,cds_phase_p,pos_mod3,marker
0,35,T,TAA,C22,C,2,2,2,
1,36,A,AAC,C00,C,0,0,0,
2,37,A,ACG,C11,C,1,1,1,
3,38,C,CGA,C22,C,2,2,2,
4,39,G,GAA,C00,C,0,0,0,
5,40,A,AAG,C11,C,1,1,1,
6,41,A,AGA,C22,C,2,2,2,
7,42,G,GAG,C00,C,0,0,0,
8,43,A,AGT,C11,C,1,1,1,DONOR
9,44,G,GTA,I22,I,2,2,2,


Junction 1
  donor pos:    72     state=C00
  intron len:   15     mod3=0
  first intron: 73     state=I11
  last intron:  87     state=I01
  acceptor pos: 88     state=C11
  carried p through intron: 1 -> 1 -> 1
  genomic g shift donor→acceptor: 0 -> 1



,pos,base,codon_from_here,state,region,genomic_g,cds_phase_p,pos_mod3,marker
0,64,A,AAG,C11,C,1,1,1,
1,65,A,AGA,C22,C,2,2,2,
2,66,G,GAG,C00,C,0,0,0,
3,67,A,AGC,C11,C,1,1,1,
4,68,G,GCC,C22,C,2,2,2,
5,69,C,CCA,C00,C,0,0,0,
6,70,C,CAC,C11,C,1,1,1,
7,71,A,ACG,C22,C,2,2,2,
8,72,C,CGT,C00,C,0,0,0,DONOR
9,73,G,GTA,I11,I,1,1,1,


EXAMPLE: intron_mod = 1
DNA length: 152
Start ATG: 24  ATG
Stop start: 125  TAG
Intron lengths: (19, 13)

Junction 0
  donor pos:    43     state=C11
  intron len:   19     mod3=1
  first intron: 44     state=I22
  last intron:  62     state=I22
  acceptor pos: 63     state=C02
  carried p through intron: 2 -> 2 -> 2
  genomic g shift donor→acceptor: 1 -> 0



...

,pos,base,codon_from_here,state,region,genomic_g,cds_phase_p,pos_mod3,marker
0,65,A,AAG,C21,C,2,1,2,
1,66,A,AGA,C02,C,0,2,0,
2,67,G,GAG,C10,C,1,0,1,
3,68,A,AGC,C21,C,2,1,2,
4,69,G,GCC,C02,C,0,2,0,
5,70,C,CCA,C10,C,1,0,1,
6,71,C,CAC,C21,C,2,1,2,
7,72,A,ACG,C02,C,0,2,0,
8,73,C,CGT,C10,C,1,0,1,DONOR
9,74,G,GTA,I21,I,2,1,2,


EXAMPLE: intron_mod = 2
DNA length: 148
Start ATG: 24  ATG
Stop start: 121  TAG
Intron lengths: (20, 8)

Junction 0
  donor pos:    43     state=C11
  intron len:   20     mod3=2
  first intron: 44     state=I22
  last intron:  63     state=I02
  acceptor pos: 64     state=C12
  carried p through intron: 2 -> 2 -> 2
  genomic g shift donor→acceptor: 1 -> 1



,pos,base,codon_from_here,state,region,genomic_g,cds_phase_p,pos_mod3,marker
0,35,T,TAA,C22,C,2,2,2,
1,36,A,AAC,C00,C,0,0,0,
2,37,A,ACG,C11,C,1,1,1,
3,38,C,CGA,C22,C,2,2,2,
4,39,G,GAA,C00,C,0,0,0,
5,40,A,AAG,C11,C,1,1,1,
6,41,A,AGA,C22,C,2,2,2,
7,42,G,GAG,C00,C,0,0,0,
8,43,A,AGT,C11,C,1,1,1,DONOR
9,44,G,GTT,I22,I,2,2,2,


Junction 1
  donor pos:    74     state=C20
  intron len:   8     mod3=2
  first intron: 75     state=I01
  last intron:  82     state=I11
  acceptor pos: 83     state=C21
  carried p through intron: 1 -> 1 -> 1
  genomic g shift donor→acceptor: 2 -> 2



,pos,base,codon_from_here,state,region,genomic_g,cds_phase_p,pos_mod3,marker
0,66,A,AAG,C01,C,0,1,0,
1,67,A,AGA,C12,C,1,2,1,
2,68,G,GAG,C20,C,2,0,2,
3,69,A,AGC,C01,C,0,1,0,
4,70,G,GCC,C12,C,1,2,1,
5,71,C,CCA,C20,C,2,0,2,
6,72,C,CAC,C01,C,0,1,0,
7,73,A,ACG,C12,C,1,2,1,
8,74,C,CGT,C20,C,2,0,2,DONOR
9,75,G,GTG,I01,I,0,1,0,
